# NMDesc escape disease-gene paper — reproducible Colab runner

Runs the R analysis in [`CobanAkdemirlab/NMDescapediseasegene_paper`](https://github.com/CobanAkdemirlab/NMDescapediseasegene_paper) (v5 directories) on Google
Colab. Nothing to install locally; no Google account data required beyond signing in to
Colab.

### What you need
Nothing. The repository carries its inputs under `data/ship` (15 files, 101 MB), so the clone in
section 1 is most of the data step. Eight further inputs are not in the repository and are
listed there; steps that need them name the missing file.

### Runtime
Python 3, CPU. R is called through `rpy2` cell magics. No GPU is used — this pipeline is
CPU-bound R.

## 1. Repository and data

The inputs ship with the repository under `data/ship` — 15 files, 101 MB, so the clone
below is most of the data step. Nothing to download separately and no Drive to mount.

Eight inputs the seven steps declare are not in `data/ship`. Four are large public downloads
with their own terms: `variant_summary.txt` (ClinVar, 3.8 GB), `clinvar_20260201.vcf.gz`
(ClinVar, 186 MB), `gnomad.v2.1.1.lof_metrics.by_gene.txt` (gnomAD, 13 MB), and `human (1).txt`,
a 174 MB interaction table of unverified provenance. Two are small and simply absent:
`omim_AD_symbols.csv` (20 KB, needed by steps 1, 2, 3 and 6) and `ptc_can_NMD_df.csv` (6.8 MB,
the only input standing between a fresh clone and a complete step 5). Two are step 1
intermediates that nothing else regenerates: `snv_plp_ptc_nmdesc_can20260201.rds` and
`snv_plp_ptc_nmdesc_can_filtered20260201.rds`.

Steps that need them report the missing file by name. To supply them, set `EXTRA_DATA_URL` to
an archive holding them, or copy them into `data/ship` yourself.

In [ ]:
REPO_URL   = "https://github.com/CobanAkdemirlab/NMDescapediseasegene_paper.git"
REPO_DIR   = "/content/NMDescapediseasegene_paper"
EXTRA_DATA_URL = ""   # optional archive with the four large public files

import os, subprocess, sys
if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "clone", "-q", "--depth", "1", REPO_URL, REPO_DIR], check=True)
DATA_DIR = os.path.join(REPO_DIR, "data", "ship")
assert os.path.isdir(DATA_DIR), f"{DATA_DIR} missing -- the clone has no data/ship"

if EXTRA_DATA_URL:
    arch = "/content/" + EXTRA_DATA_URL.rstrip("/").split("/")[-1].split("?")[0]
    if not os.path.exists(arch):
        subprocess.run(["wget", "-q", "--show-progress", "-O", arch, EXTRA_DATA_URL], check=True)
    if arch.endswith((".tar.gz", ".tgz")):
        subprocess.run(["tar", "xzf", arch, "-C", DATA_DIR], check=True)
    elif arch.endswith(".zip"):
        subprocess.run(["unzip", "-q", "-o", arch, "-d", DATA_DIR], check=True)
    else:
        sys.exit("EXTRA_DATA_URL must be .tar.gz or .zip")

# Flatten, so a nested archive layout still resolves by basename
n = 0
for dp, dn, fn in os.walk(DATA_DIR):
    if dp == DATA_DIR: continue
    for f in fn:
        dst = os.path.join(DATA_DIR, f)
        if not os.path.exists(dst):
            os.link(os.path.join(dp, f), dst); n += 1
print("commit :", subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"],
                                 capture_output=True, text=True).stdout.strip())
print("data   :", DATA_DIR)
print("files  :", len([f for f in os.listdir(DATA_DIR)
                       if os.path.isfile(os.path.join(DATA_DIR, f))]),
      f"({n} linked up from subdirs)")

## 2. R packages

CRAN packages install from a **dated** Posit Package Manager snapshot, so the versions
resolved here are the same ones a reader gets next year. Change `SNAPSHOT` to move the
pin. Bioconductor is pinned by its own release, recorded in `sessionInfo()` at the end.

Two heavy dependencies are commented out; uncomment only if you run the stages that need
them. `BSgenome.Hsapiens.UCSC.hg38` is a ~700 MB download used by 4 scripts, and `brms`
pulls the Stan toolchain, also used by 4 scripts.

In [ ]:
SNAPSHOT = "2026-08-01"   # Posit PPM snapshot date; pins all CRAN versions
!apt-get -qq install -y r-base-core libcurl4-openssl-dev libxml2-dev libssl-dev libfontconfig1-dev > /dev/null
%load_ext rpy2.ipython
codename = subprocess.run(['bash','-lc','lsb_release -cs'],
                          capture_output=True, text=True).stdout.strip()
print("Ubuntu:", codename, "| CRAN snapshot:", SNAPSHOT)

In [ ]:
%%R -i codename -i SNAPSHOT
options(repos = c(CRAN = sprintf(
          "https://packagemanager.posit.co/cran/__linux__/%s/%s", codename, SNAPSHOT)),
        Ncpus = parallel::detectCores())
cran <- c("dplyr","ggplot2","tidyverse","readr","stringr","ggpubr","patchwork","tidyr",
          "data.table","scales","gt","gtsummary","readxl","lme4","lmerTest","rstatix",
          "flextable","here","DiscreteFDR","jsonlite","remotes","BiocManager")
need <- setdiff(cran, rownames(installed.packages()))
if (length(need)) install.packages(need)
still <- setdiff(cran, rownames(installed.packages()))
cat(if (length(still)) paste("CRAN FAILED:", paste(still, collapse=", ")) else
    "all CRAN packages present", "\n")

In [ ]:
%%R
bioc <- c("biomaRt","GenomicRanges","IRanges","S4Vectors","Biostrings","AnnotationDbi",
          "GenomicFeatures","VariantAnnotation","STRINGdb")
need <- setdiff(bioc, rownames(installed.packages()))
if (length(need)) BiocManager::install(need, ask = FALSE, update = FALSE)
still <- setdiff(bioc, rownames(installed.packages()))
cat("Bioconductor", as.character(BiocManager::version()), "|",
    if (length(still)) paste("FAILED:", paste(still, collapse=", ")) else "all present", "\n")

# aenmd is not on CRAN or Bioconductor: 3 scripts call library(aenmd), 11 use aenmd::
if (!requireNamespace("aenmd", quietly = TRUE)) {
  remotes::install_github("kostkalab/aenmd.data.ensdb.v105", upgrade = "never")
  remotes::install_github("kostkalab/aenmd", upgrade = "never")
}
cat("aenmd:", requireNamespace("aenmd", quietly = TRUE), "\n")

# BiocManager::install("BSgenome.Hsapiens.UCSC.hg38", ask = FALSE, update = FALSE)
# install.packages("brms")

## 2. Path layer

The stage scripts resolve their inputs through `lib/paths.R`, which looks files up by
name under `NMDESC_DATA`. Older scripts outside the seven steps still carry absolute
paths from the original authors' machines (`~/Desktop/...`, `/Users/jxu14/Desktop/...`);
`run_script()` substitutes those with `DATA_DIR` on a working copy and sources the copy,
leaving the checkout untouched. The seven steps go through `run_in_place()` instead,
which needs no substitution.

In [ ]:
os.environ["NMDESC_DATA"] = DATA_DIR
os.environ["NMDESC_REPO"] = REPO_DIR
os.environ["NMDESC_WORK"] = "/content/work"
print("NMDESC_DATA:", os.environ["NMDESC_DATA"])
print("NMDESC_REPO:", os.environ["NMDESC_REPO"])

In [ ]:
shim = r'''
# colab_paths.R -- path mapping layer for running this repo on Google Colab.
# The v4 scripts read data through absolute paths on the original author's
# machine. DATA_DIR points at one folder holding those inputs; run_script()
# rewrites the prefixes on a working copy and sources that copy.

DATA_DIR <- Sys.getenv("NMDESC_DATA", "/content/drive/MyDrive/NMDesc_data")
REPO_DIR <- Sys.getenv("NMDESC_REPO", "/content/NMDescapediseasegene_paper")
WORK_DIR <- Sys.getenv("NMDESC_WORK", "/content/work")
dir.create(WORK_DIR, showWarnings = FALSE, recursive = TRUE)

# Longest prefixes first, so nested ones are not shadowed.
PREFIXES <- c(
  "~/Desktop/NMDescapediseasegene_paper-main/new_NMDesc/data",
  "/Users/jxu14/Desktop/NMDescapediseasegene_paper-main",
  "~/Desktop/new_clinvar/raw_data",
  "~/Desktop/new_clinvar/snv_list/list4",
  "~/Desktop/new_clinvar",
  "/Users/jxu14/Desktop/enrich",
  "/Users/jxu14/Desktop/autism",
  "/Users/qkelly/Desktop/clinvar",
  "~/Desktop/clinvar",
  "~/Downloads",
  "~/Desktop",
  "/Users/jxu14/Desktop",
  "/Users/qkelly/Desktop"
)

map_path <- function(x) {
  for (p in PREFIXES) {
    if (startsWith(x, p)) return(file.path(DATA_DIR, sub("^/+", "", substring(x, nchar(p) + 1))))
  }
  x
}

# Every data reference a script makes, resolved against DATA_DIR.
script_inputs <- function(script) {
  txt <- readLines(file.path(REPO_DIR, script), warn = FALSE)
  m <- regmatches(txt, gregexpr('"[^"]*\\.(csv|txt|tsv|rds|fasta|fa|xlsx)"', txt, ignore.case = TRUE))
  refs <- unique(gsub('"', "", unlist(m)))
  if (!length(refs)) return(data.frame(ref = character(), resolved = character(), exists = logical()))
  res <- vapply(refs, function(r)
    if (startsWith(r, "/") || startsWith(r, "~")) map_path(r) else file.path(DATA_DIR, basename(r)),
    character(1))
  data.frame(ref = refs, resolved = unname(res), exists = file.exists(unname(res)),
             row.names = NULL, stringsAsFactors = FALSE)
}

# Report what a script needs before spending time on it.
check_script <- function(script) {
  d <- script_inputs(script)
  cat(sprintf("%s: %d data refs, %d present, %d missing\n",
              script, nrow(d), sum(d$exists), sum(!d$exists)))
  if (any(!d$exists)) print(d[!d$exists, c("ref", "resolved")], row.names = FALSE)
  invisible(d)
}

# Rewrite absolute prefixes on a copy, then source the copy.
run_script <- function(script, echo = TRUE) {
  src <- file.path(REPO_DIR, script)
  txt <- readLines(src, warn = FALSE)
  for (p in PREFIXES) txt <- gsub(p, DATA_DIR, txt, fixed = TRUE)
  txt <- gsub("setwd\\(", "# setwd(", txt)
  dst <- file.path(WORK_DIR, gsub("[/ ]", "_", script))
  writeLines(txt, dst)
  source(dst, echo = echo, max.deparse.length = 200)
  invisible(dst)
}

# Sources a script where it lives, with the working directory set to its own
# directory, so relative helper lookups inside the script resolve. Use this for
# entry points; run_script() suits scripts that carry hardcoded data paths.
run_in_place <- function(script, echo = FALSE) {
  path <- file.path(REPO_DIR, script)
  wd <- getwd(); on.exit(setwd(wd), add = TRUE)
  setwd(dirname(path))
  source(basename(path), echo = echo, max.deparse.length = 200)
  invisible(path)
}

cat("colab_paths.R loaded\n  DATA_DIR:", DATA_DIR, "\n  REPO_DIR:", REPO_DIR, "\n")
'''
open('/content/colab_paths.R','w').write(shim)
print(len(shim.splitlines()), 'lines written')

In [ ]:
%%R
source("/content/colab_paths.R")

## 4. Readiness audit

Resolves every data reference each script makes and reports which inputs are present.
Run this before any stage — a missing input is the usual reason one fails.

In [ ]:
%%R
scripts <- list.files(REPO_DIR, pattern = "[.][Rr]$", recursive = TRUE)
scripts <- scripts[!grepl("^backup/", scripts)]
rdy <- do.call(rbind, lapply(scripts, function(s) {
  d <- script_inputs(s)
  data.frame(script = s, n_ref = nrow(d), n_missing = sum(!d$exists))
}))
rdy$ready <- rdy$n_ref > 0 & rdy$n_missing == 0
cat("repository scripts:", nrow(rdy),
    "| runnable now:", sum(rdy$ready),
    "| blocked:", sum(rdy$n_ref > 0 & !rdy$ready),
    "| no data refs:", sum(rdy$n_ref == 0), "\n\n")
print(head(rdy[order(-rdy$ready, rdy$n_missing), ], 15), row.names = FALSE)
write.csv(rdy, "/content/work/readiness.csv", row.names = FALSE)

In [ ]:
%%R
# Which inputs are still missing, ranked by how many scripts want them
miss <- do.call(rbind, lapply(scripts, function(s) {
  d <- script_inputs(s); d <- d[!d$exists, , drop = FALSE]
  if (!nrow(d)) NULL else data.frame(need = basename(d$ref))
}))
if (is.null(miss)) cat("nothing missing\n") else print(head(sort(table(miss$need),
                                                                 decreasing = TRUE), 30))

## 5. Run the analysis

`run_analysis.R` at the repository root drives seven steps in order:

1. `gene level_v5/gene_get_main.R` — SNV disease genes (ClinVar VCF → NMDesc enrichment)
2. `gene level_v5/disease genes/framesift/fs_transcript_level.R`, `bind_result_dbh.R` — FS disease genes (per-transcript ACAT / BH)
3. `gene level_v5/control genes/snv/get_snv_control_gene.R`, `control genes/frameshift/get_fs_control_gene.R` — AD-restricted control gene lists
4. `variant level_v5/variant_get_main.R` — ClinVar P/LP disease variant lists
5. `variant level_v5/variant_get_main.R` — gnomAD control variant lists
6. `gene level_v5/gene_compare_main.R` — gene-level comparison, CDS-matched pairs
7. `variant level_v5/variant_compare_main.R` — variant-level comparison (GLM / GLMM / Bayesian)

Each step is located by file name inside the repository, so a script that moves between
directories still runs. `run_analysis.R` sources `gene level_v5/lib/paths.R` so `data_file()`
resolves inputs by name under `NMDESC_DATA`, and reports each step separately: a step whose
declared outputs already exist is `cached`, one with unmet inputs or packages is `blocked`,
and the chain continues either way.

`Rscript run_analysis.R --dry-run` reports every step's status without running anything, and
`Rscript run_analysis.R 6 7` runs a subset.

It is sourced in place: `run_script()` copies a script into the work directory and comments
out `setwd()`, which removes the per-step directory switching the step scripts rely on.


In [ ]:
%%R
check_script("run_analysis.R")

In [ ]:
%%R
txt <- readLines(file.path(REPO_DIR, "run_analysis.R"), warn = FALSE)
want <- c("gene_get_main.R", "fs_transcript_level.R", "bind_result_dbh.R",
          "get_snv_control_gene.R", "get_fs_control_gene.R", "variant_get_main.R",
          "gene_compare_main.R", "variant_compare_main.R")
miss <- want[!vapply(want, function(s) any(grepl(s, txt, fixed = TRUE)), logical(1))]
if (length(miss)) stop("run_analysis.R does not list: ", paste(miss, collapse = ", "))
run_in_place("run_analysis.R")


## 6. Outputs and the environment record

Colab discards `/content` when the runtime recycles. `sessionInfo()` is written next to
the outputs so the exact package versions behind a result stay attached to it.

In [ ]:
%%R
out <- "/content/work/output"; dir.create(out, showWarnings = FALSE, recursive = TRUE)
made <- setdiff(list.files(WORK_DIR, pattern = "[.](csv|pdf|png|rds|txt)$",
                           recursive = TRUE, full.names = TRUE),
                list.files(out, recursive = TRUE, full.names = TRUE))
if (length(made)) file.copy(made, out, overwrite = TRUE)
writeLines(capture.output(sessionInfo()), file.path(out, "sessionInfo.txt"))
cat("outputs:", length(made), "->", out, "\n")

In [ ]:
from google.colab import files
!cd /content/work && tar czf /content/nmdesc_output.tar.gz output
files.download('/content/nmdesc_output.tar.gz')